In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os 

In [ ]:
GRAPH_LOG_DIR = 'logs/paper_graph_logs'

dfs = {}
for file in os.listdir(GRAPH_LOG_DIR):
    if file.endswith('_iterations.csv'):
        name = file.replace('_iterations.csv', '')
        dfs[name] = pd.read_csv(os.path.join(GRAPH_LOG_DIR, file))

print(f'Loaded {len(dfs)} iteration logs from {GRAPH_LOG_DIR}')

In [ ]:
for name, df in dfs.items():
    print(df['break_op'].unique())

In [ ]:
names = sorted(dfs.keys())
cols = 3
rows = -(-len(names) // cols)

all_ops = ['tool', 'vehicle', 'vehicle_days', 'distance', 'worst_day', 'random', 'geo']
wb_cols = {
    'tool': 'wb_tool',
    'vehicle': 'wb_vehicle',
    'vehicle_days': 'wb_veh_days',
    'distance': 'wb_distance',
    'worst_day': 'wb_worst',
    'random': 'wb_random',
    'geo': 'wb_geo',
}

fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten()

legend_h1, legend_l1 = [], []
legend_h2, legend_l2 = [], []

for i, name in enumerate(names):
    ax = axes[i]
    df = dfs[name]

    for op in all_ops:
        col = wb_cols[op]
        if col in df.columns:
            ax.plot(df['iteration'], df[col], label=op)

    ax2 = ax.twinx()
    if 'current_cost' in df.columns:
        ax2.plot(df['iteration'], df['current_cost'], color='0.55', alpha=0.35, linewidth=1.2, label='current_cost')
    if 'best_cost' in df.columns:
        ax2.plot(df['iteration'], df['best_cost'], color='0.15', alpha=0.55, linewidth=1.4, linestyle='--', label='best_cost')

    # Keep weights visually in front, cost lines in the background
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

    ax.set_title(name.replace('challenge_', ''), fontsize=10)
    ax.set_xlabel('Iteration', fontsize=8)
    ax.set_ylabel('Weight', fontsize=8)
    ax2.set_ylabel('Cost', fontsize=8, color='0.3')
    ax2.tick_params(axis='y', labelsize=7, colors='0.35')

    if i == 0:
        legend_h1, legend_l1 = ax.get_legend_handles_labels()
        legend_h2, legend_l2 = ax2.get_legend_handles_labels()

for i in range(len(names), len(axes)):
    axes[i].set_visible(False)

# One shared, larger legend on top for paper readability
fig.legend(
    legend_h1 + legend_h2,
    legend_l1 + legend_l2,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.02),
    ncol=5,
    fontsize=10,
    frameon=True,
    borderpad=0.6,
    columnspacing=1.2,
    handlelength=2.0,
    handletextpad=0.6,
    fancybox=True,
    shadow=False,
    title='Operators and Cost Curves',
    title_fontsize=11,
 )
plt.tight_layout(rect=[0, 0, 1, 0.94])

In [ ]:
from pathlib import Path
import pandas as pd

logs_root = Path("logs")
csv_paths = sorted(logs_root.rglob("*_iterations.csv"))

records = []
missing_best_cost = []
read_errors = {}

for csv_path in csv_paths:
    stem = csv_path.stem
    challenge = stem[: -len("_iterations")] if stem.endswith("_iterations") else stem

    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        read_errors[str(csv_path)] = str(exc)
        continue

    if "best_cost" not in df.columns:
        missing_best_cost.append(str(csv_path))
        continue

    series = pd.to_numeric(df["best_cost"], errors="coerce").dropna()
    if series.empty:
        continue

    records.append(
        {
            "challenge": challenge,
            "last_best_cost": series.iloc[-1],
            "file": str(csv_path),
            "is_paper_graph_logs": "paper_graph_logs" in str(csv_path),
            "path_depth": len(csv_path.parts),
        }
    )

long_df = pd.DataFrame(records)

if not long_df.empty:
    # If there are duplicates per challenge, prefer non-paper_graph_logs and shorter paths.
    comparison_df = (
        long_df.sort_values(["challenge", "is_paper_graph_logs", "path_depth", "file"])
.drop_duplicates(subset=["challenge"], keep="first")
        .loc[:, ["challenge", "last_best_cost"]]
        .sort_values("challenge")
        .reset_index(drop=True)
    )
else:
    comparison_df = pd.DataFrame(columns=["challenge", "last_best_cost"])

print(f"CSV files found: {len(csv_paths)}")
print(f"Files parsed with best_cost: {len(long_df)}")
print(f"Unique challenges: {comparison_df['challenge'].nunique() if not comparison_df.empty else 0}")
print(f"Files missing best_cost: {len(missing_best_cost)}")
print(f"Files with read errors: {len(read_errors)}")

if comparison_df.empty:
    print("No best_cost values found.")
else:
    display(comparison_df)

# Optional details are available in long_df if needed.

In [ ]:
cost_shares = []
selection_shares = []
labels = []

cost_ops = ['tool', 'vehicle', 'vehicle_days', 'distance']

for name, df in dfs.items():
    breakdown = [df['bd_tool'].mean(), df['bd_vehicle'].mean(), 
                 df['bd_veh_days'].mean(), df['bd_distance'].mean()]
    breakdown_pct = [b / sum(breakdown) for b in breakdown]
    
    cost_df = df[df['break_op'].isin(cost_ops)]
    counts = cost_df['break_op'].value_counts()
    total = counts.sum()
    sel = [counts.get(comp, 0) / total for comp in cost_ops]
    
    for comp, cs, ss in zip(cost_ops, breakdown_pct, sel):
        cost_shares.append(cs)
        selection_shares.append(ss)
        labels.append(f'{name[-6:]}/{comp}')

plt.figure(figsize=(10, 10))
plt.scatter(cost_shares, selection_shares, alpha=0.6)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect tracking')

for i, label in enumerate(labels):
    # Only label outliers far from diagonal
    diff = abs(cost_shares[i] - selection_shares[i])
    if diff > 0.3:
        plt.annotate(label, (cost_shares[i], selection_shares[i]), fontsize=7)

plt.xlabel('Cost component share')
plt.ylabel('Operator selection frequency (cost-targeted only)')
plt.legend()
plt.xlim(0, 1)
plt.ylim(0, 1)

In [ ]:
all_ops = ['tool', 'vehicle', 'vehicle_days', 'distance', 'worst_day', 'random', 'geo']
wb_cols = ['wb_tool', 'wb_vehicle', 'wb_veh_days', 'wb_distance', 'wb_worst', 'wb_random', 'wb_geo']
bd_cols = ['bd_tool', 'bd_vehicle', 'bd_veh_days', 'bd_distance']

names = sorted(dfs.keys())
cols = 3
rows = -(-len(names) // cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
axes = axes.flatten()

for i, name in enumerate(names):
    ax = axes[i]
    df = dfs[name]
    final = df.iloc[-1]

    # 1) Cost share: only defined for the four cost-targeted operators
    bd = [final[c] for c in bd_cols]
    bd_pct = [b / sum(bd) * 100 for b in bd]

    # 2) Weight share across all break operators
    wb = [final[c] for c in wb_cols]
    wb_pct = [w / sum(wb) * 100 for w in wb]

    # 3) Actual selection frequency across all break operators
    counts = df['break_op'].value_counts()
    total = counts.sum()
    sel_pct = [counts.get(op, 0) / total * 100 for op in all_ops]

    x = list(range(len(all_ops)))
    w = 0.25

    # Plot cost share only on first four operator positions
    bars_cost = ax.bar([j - w for j in x[:4]], bd_pct, w, label='Cost share (cost ops only)', color='tab:blue')
    bars_weight = ax.bar([j for j in x], wb_pct, w, label='Weight share', color='tab:red')
    bars_sel = ax.bar([j + w for j in x], sel_pct, w, label='Actual selection', color='tab:green')

    ax.set_xticks(x)
    ax.set_xticklabels(['tool', 'veh', 'v-days', 'dist', 'worst', 'rand', 'geo'], fontsize=8)
    ax.set_title(name.replace('challenge_', ''), fontsize=10)
    ax.set_ylim(0, 100)

for i in range(len(names), len(axes)):
    axes[i].set_visible(False)

# One shared, larger legend on top for paper readability
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    fontsize=10,
    frameon=True,
    borderpad=0.6,
    columnspacing=1.2,
    handlelength=2.0,
    handletextpad=0.6,
    fancybox=True,
    shadow=False,
    title='Share Types',
    title_fontsize=11,
 )
plt.tight_layout(rect=[0, 0, 1, 0.94])

In [ ]:
from scipy.stats import pearsonr

cost_shares = []
selection_shares = []

cost_ops = ['tool', 'vehicle', 'vehicle_days', 'distance']

for name, df in dfs.items():
    breakdown = [df['bd_tool'].mean(), df['bd_vehicle'].mean(), 
                 df['bd_veh_days'].mean(), df['bd_distance'].mean()]
    breakdown_pct = [b / sum(breakdown) for b in breakdown]
    
    cost_df = df[df['break_op'].isin(cost_ops)]
    counts = cost_df['break_op'].value_counts()
    total = counts.sum()
    sel = [counts.get(comp, 0) / total for comp in cost_ops]
    
    cost_shares.extend(breakdown_pct)
    selection_shares.extend(sel)

r, p = pearsonr(cost_shares, selection_shares)
print(f'Pearson r = {r:.3f}, p = {p:.6f}')

In [ ]:
from scipy.stats import pearsonr
import numpy as np

all_bd = []
all_wb = []
all_sel = []

# Keep this comparison strictly on cost-targeted operators
cost_ops = ['tool', 'vehicle', 'vehicle_days', 'distance']
wb_cost_cols = ['wb_tool', 'wb_vehicle', 'wb_veh_days', 'wb_distance']

for name, df in dfs.items():
    final = df.iloc[-1]

    bd = [final[c] for c in bd_cols]
    bd_pct = [b / sum(bd) for b in bd]

    wb = [final[c] for c in wb_cost_cols]
    wb_pct = [w / sum(wb) for w in wb]

    cost_df = df[df['break_op'].isin(cost_ops)]
    counts = cost_df['break_op'].value_counts()
    total = counts.sum()
    sel = [counts.get(comp, 0) / total for comp in cost_ops]

    all_bd.extend(bd_pct)
    all_wb.extend(wb_pct)
    all_sel.extend(sel)

r_vanilla, _ = pearsonr(all_bd, all_wb)
r_ours, _ = pearsonr(all_bd, all_sel)

mae_vanilla = np.mean(np.abs(np.array(all_bd) - np.array(all_wb)))
mae_ours = np.mean(np.abs(np.array(all_bd) - np.array(all_sel)))

print(f'Correlation with cost share:')
print(f'  Vanilla (weight only):  r = {r_vanilla:.3f}')
print(f'  Ours (weight × cost):   r = {r_ours:.3f}')
print(f'')
print(f'Mean absolute error from cost share:')
print(f'  Vanilla (weight only):  MAE = {mae_vanilla:.3f}')
print(f'  Ours (weight × cost):   MAE = {mae_ours:.3f}')

## Complexity (Big-O + model size)

This cell set gives complexity for the implemented algorithm and model-size counts for the exact CP-SAT fallback model.

- Routing is solved with OR-Tools Routing (CP-based search), not a classical MIP like Gurobi.
- Feasibility repair uses CP-SAT and has explicit integer variables/constraints we can count.
- Runtime section below uses existing files in `logs/` only (no re-run), so it is fast.

In [ ]:
import re
import json
from pathlib import Path
import pandas as pd

instance_files = sorted(Path('instances').glob('challenge_*.txt'))

rows = []
for p in instance_files:
    m = re.search(r'r(\d+)d(\d+)', p.stem)
    if not m:
        continue
    n = int(m.group(1))
    d = int(m.group(2))
    s = 2 * n  # delivery + pickup stops
    avg_s_day = s / d if d else float('nan')

    # Data-prep cost in routing/solver.py builds dense distance matrices per day: O(s_day^2).
    # Across all days this is O(sum_d s_d^2), worst-case O((2n)^2), balanced O(4n^2/d).
    rows.append({
        'instance': p.stem,
        'n_requests (n)': n,
        'days (d)': d,
        'stops (2n)': s,
        'avg_stops/day': round(avg_s_day, 2),
        'prep_matrix_cost_balanced': f'O(4*n^2/d) ~ O({round(4*n*n/d) if d else "nan"})',
        'prep_matrix_cost_worst': f'O(n^2) ~ O({(2*n)*(2*n)})',
    })

complexity_df = pd.DataFrame(rows).sort_values('instance').reset_index(drop=True)
display(complexity_df)

print('--- Complexity summary used in report ---')
print('1) ALNS outer loop with I iterations:')
print('   O(I * (destroy + repair + reroute(changed_days)))')
print('   Dominant controlled term is rerouting prep O(sum_d s_d^2), upper-bounded by O(n^2) per iteration.')
print('   So practical upper expression: O(I * n^2) + solver-search cost (problem dependent).')
print('')
print('2) CP-SAT feasibility repair model size for one tool type with n_t requests:')
print('   - Integer vars: 2*n_t  (start and end per request)')
print('   - Interval vars: n_t')
print('   - Explicit linear constraints: n_t (horizon bounds) + 1 cumulative resource constraint')
print('   - Hints: up to n_t (do not change feasibility region)')
print('')
print('3) OR-Tools routing model: CP routing search (not MIP in this code), so no fixed MILP var/constraint count is exposed.')

In [ ]:
from pathlib import Path
import re
import json
import time
import subprocess
import sys
from datetime import datetime
import pandas as pd

log_dir = Path('logs')
csv_logs = sorted(log_dir.glob('challenge_*_iterations.csv'))
bench_file = log_dir / 'runtime_benchmarks.jsonl'

def run_and_record_benchmarks(
    instances,
    method='alns',
    repeats=1,
    timeout_sec=None,
    run_label='manual',
    keep_existing=True,
    dry_run=True,
):
    
    existing = []
    if bench_file.exists():
        with open(bench_file, 'r', encoding='utf-8') as fh:
            for line in fh:
                line = line.strip()
                if line:
                    existing.append(json.loads(line))

    existing_keys = {(e['instance'], e.get('method', 'alns')) for e in existing}
    to_run = []
    for inst in instances:
        key = (inst, method)
        if keep_existing and key in existing_keys:
            continue
        to_run.append(inst)

    print(f'Planned runs: {len(to_run)} (dry_run={dry_run})')
    if dry_run:
        return pd.DataFrame(existing) if existing else pd.DataFrame()

    records = []
    for inst in to_run:
        instance_file = Path('instances') / f'{inst}.txt'
        if not instance_file.exists():
            print(f'SKIP missing instance file: {instance_file}')
            continue

        for rep in range(repeats):
            cmd = [sys.executable, 'main.py', str(instance_file), '--method', method]
            t0 = time.perf_counter()
            rc = -999
            timed_out = False
            try:
                out = subprocess.run(
                    cmd,
                    capture_output=True,
                    text=True,
                    timeout=timeout_sec,
                    check=False,
                )
                rc = out.returncode
                tail = '\n'.join(out.stdout.splitlines()[-5:]) if out.stdout else ''
            except subprocess.TimeoutExpired as ex:
                timed_out = True
                rc = -1
                tail = f'TIMEOUT: {ex}'

            dt = time.perf_counter() - t0
            rec = {
                'timestamp_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
                'instance': inst,
                'method': method,
                'repeat': rep + 1,
                'wall_time_sec': dt,
                'returncode': rc,
                'timed_out': timed_out,
                'run_label': run_label,
                'stdout_tail': tail,
            }
            records.append(rec)
            print(f"{inst} rep {rep + 1}/{repeats}: {dt:.2f}s rc={rc} timeout={timed_out}")

    if records:
        with open(bench_file, 'a', encoding='utf-8') as fh:
            for rec in records:
                fh.write(json.dumps(rec) + '\n')
    
    all_records = existing + records
    return pd.DataFrame(all_records) if all_records else pd.DataFrame()

def load_exact_runtime_records():
    if not bench_file.exists():
        return pd.DataFrame()
    rows = []
    with open(bench_file, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

# --- Mode A: exact runtime records (preferred) ---
exact_df = load_exact_runtime_records()

if not exact_df.empty:
    m = exact_df['instance'].str.extract(r'r(\d+)d(\d+)')
    exact_df['family'] = 'r' + m[0].fillna('unknown') + 'd' + m[1].fillna('unknown')

    # Keep successful, non-timeout runs for stats
    exact_ok = exact_df[(exact_df['returncode'] == 0) & (~exact_df['timed_out'])].copy()

    print('--- Exact runtime records found (from runtime_benchmarks.jsonl) ---')
    display(exact_ok[['timestamp_utc', 'instance', 'family', 'method', 'repeat', 'wall_time_sec', 'run_label']])

    if not exact_ok.empty:
        print('\n--- Exact runtime stats by instance ---')
        by_instance = exact_ok.groupby('instance')['wall_time_sec'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).sort_index()
        display(by_instance)

        print('\n--- Exact runtime stats by challenge family ---')
        by_family_exact = exact_ok.groupby('family')['wall_time_sec'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).sort_index()
        display(by_family_exact)
    else:
        print('No successful exact runs yet. Use run_and_record_benchmarks(..., dry_run=False).')

else:
    print('No exact runtime benchmark file yet (logs/runtime_benchmarks.jsonl).')

    # --- Mode B: fallback estimate from file timestamps ---
    runtime_rows = []
    for csv_path in csv_logs:
        instance = csv_path.name.replace('_iterations.csv', '')
        m = re.search(r'r(\d+)d(\d+)', instance)
        family = f"r{m.group(1)}d{m.group(2)}" if m else 'unknown'

        st = csv_path.stat()
        runtime_sec_est = max(0.0, st.st_mtime - st.st_ctime)

        summary_path = log_dir / f'{instance}_summary.json'
        iterations = None
        best_cost = None
        stop_reason = None
        if summary_path.exists():
            with open(summary_path, 'r', encoding='utf-8') as fh:
                s = json.load(fh)
            iterations = s.get('iterations_run')
            best_cost = s.get('final_best_cost')
            stop_reason = s.get('stop_reason')

        iter_per_sec_est = None
        if iterations and runtime_sec_est > 0:
            iter_per_sec_est = iterations / runtime_sec_est

        runtime_confidence = 'high' if runtime_sec_est >= 5 else 'low'

        runtime_rows.append({
            'instance': instance,
            'family': family,
            'iterations': iterations,
            'runtime_sec_est': runtime_sec_est,
            'runtime_min_est': runtime_sec_est / 60.0,
            'iter_per_sec_est': iter_per_sec_est,
            'final_best_cost': best_cost,
            'stop_reason': stop_reason,
            'runtime_confidence': runtime_confidence,
            'runtime_source': 'timestamps(iterations.csv)',
        })

    runtime_df = pd.DataFrame(runtime_rows).sort_values('instance').reset_index(drop=True)
    display(runtime_df)

    print('--- Overall runtime statistics (estimated) ---')
    if not runtime_df.empty:
        overall = runtime_df['runtime_sec_est'].describe(percentiles=[0.25, 0.5, 0.75])
        print(overall.to_string())

        print('\n--- Runtime by challenge family (estimated) ---')
        by_family_all = runtime_df.groupby('family')['runtime_sec_est'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        display(by_family_all)

        print('\n--- Iteration speed statistics (estimated) ---')
        ips = runtime_df['iter_per_sec_est'].dropna()
        if len(ips) > 0:
            print(ips.describe(percentiles=[0.25, 0.5, 0.75]).to_string())
        else:
            print('No iteration-speed estimate available.')
    else:
        print('No challenge iteration logs found in logs/.')

print('\nHow to collect better runtimes now:')
print("1) dry-run plan: run_and_record_benchmarks(['challenge_r200d15_1','challenge_r300d20_4'], dry_run=True)")
print("2) run exact timing and persist: run_and_record_benchmarks(['challenge_r200d15_1','challenge_r300d20_4'], repeats=2, dry_run=False)")
print("3) then rerun this cell to get exact statistics from logs/runtime_benchmarks.jsonl")

## Exact Runtime Collection (optional)

Use this cell to run selected instances with wall-clock timing and persist results to `logs/runtime_benchmarks.jsonl`.

Set `RUN_BENCHMARKS = True` only when you want to execute (can take long).

In [ ]:
# Toggle this to run exact timing benchmarks
RUN_BENCHMARKS = True

# Choose the instances you care about (add challenge_r200d15_4 if/when its instance file exists)
TARGET_INSTANCES = [
    'challenge_r200d15_1',
    'challenge_r300d20_1',
    'challenge_r300d20_2',
    'challenge_r300d20_3',
    'challenge_r300d20_4',
    'challenge_r300d20_5',
    'challenge_r500d25_1',
    'challenge_r500d25_2',
    'challenge_r500d25_3',
]

# Suggested settings to keep runs manageable
REPEATS = 1
TIMEOUT_SEC = None      # e.g. 5400 for 90-min cap per run
KEEP_EXISTING = True    # do not rerun instances already recorded
RUN_LABEL = 'analysis_notebook'

print(f'Benchmark plan: {len(TARGET_INSTANCES)} instances, repeats={REPEATS}, run={RUN_BENCHMARKS}')

# Always show the plan first
_ = run_and_record_benchmarks(
    TARGET_INSTANCES,
    repeats=REPEATS,
    timeout_sec=TIMEOUT_SEC,
    keep_existing=KEEP_EXISTING,
    run_label=RUN_LABEL,
    dry_run=True,
)

if RUN_BENCHMARKS:
    exact_records = run_and_record_benchmarks(
        TARGET_INSTANCES,
        repeats=REPEATS,
        timeout_sec=TIMEOUT_SEC,
        keep_existing=KEEP_EXISTING,
        run_label=RUN_LABEL,
        dry_run=False,
    )
    print('\nDone. Re-run the runtime statistics cell above to refresh exact values.')
    if not exact_records.empty:
        display(exact_records.tail(10))
else:
    print('Set RUN_BENCHMARKS = True to execute and persist exact wall-clock runtimes.')

## Method comparison: heuristics vs ALNS

> This section compares final objective values and runtime across methods:
>
> - `alns` (cost-guided operator selection)
> - greedy baselines (`greedy_gls` and `greedy_*_gls`)
>
> It uses persisted solution files in `solutions/<method>/...` and exact benchmark records in `logs/runtime_benchmarks.jsonl` when available.

In [ ]:
from pathlib import Path
from main import METHODS

# Optional: run exact wall-clock benchmarks for multiple methods
RUN_METHOD_COMPARISON_BENCHMARKS = False

COMPARE_METHODS = [
    'alns',
    'greedy_gls',
    'greedy_edd_gls',
    'greedy_tight_gls',
    'greedy_heavy_gls',
    'greedy_late_gls',
]
COMPARE_METHODS = [m for m in COMPARE_METHODS if m in METHODS]

COMPARE_INSTANCES = [
    'challenge_r200d15_1',
    'challenge_r300d20_1',
    'challenge_r300d20_2',
    'challenge_r300d20_3',
    'challenge_r300d20_4',
    'challenge_r300d20_5',
    'challenge_r500d25_1',
    'challenge_r500d25_2',
    'challenge_r500d25_3',
]

CMP_REPEATS = 1
CMP_TIMEOUT_SEC = None
CMP_KEEP_EXISTING = True
CMP_RUN_LABEL = 'method_comparison'

print(f'Method benchmark plan: {len(COMPARE_METHODS)} methods x {len(COMPARE_INSTANCES)} instances')
for m in COMPARE_METHODS:
    print(f'  - {m}')
    _ = run_and_record_benchmarks(
        COMPARE_INSTANCES,
        method=m,
        repeats=CMP_REPEATS,
        timeout_sec=CMP_TIMEOUT_SEC,
        keep_existing=CMP_KEEP_EXISTING,
        run_label=CMP_RUN_LABEL,
        dry_run=True,
    )

if RUN_METHOD_COMPARISON_BENCHMARKS:
    for m in COMPARE_METHODS:
        print(f'\nRunning exact benchmarks for method: {m}')
        _ = run_and_record_benchmarks(
            COMPARE_INSTANCES,
            method=m,
            repeats=CMP_REPEATS,
            timeout_sec=CMP_TIMEOUT_SEC,
            keep_existing=CMP_KEEP_EXISTING,
            run_label=CMP_RUN_LABEL,
            dry_run=False,
        )
    print('\nDone. Re-run the next cell to refresh method comparison tables/plots.')
else:
    print('Set RUN_METHOD_COMPARISON_BENCHMARKS = True to execute full multi-method benchmarking.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def read_solution_cost(solution_path: Path):
    if not solution_path.exists():
        return None
    try:
        with open(solution_path, 'r', encoding='utf-8') as fh:
            for line in fh:
                if line.startswith('COST ='):
                    return int(line.split('=', 1)[1].strip())
    except Exception:
        return None
    return None

def collect_solution_scores(instances, methods):
    rows = []
    for method in methods:
        for inst in instances:
            sol_path = Path('solutions') / method / f'{inst}_solution.txt'
            cost = read_solution_cost(sol_path)
            rows.append({
                'instance': inst,
                'method': method,
                'cost': cost,
                'has_solution': cost is not None,
            })
    return pd.DataFrame(rows)

score_df = collect_solution_scores(COMPARE_INSTANCES, COMPARE_METHODS)

if score_df['has_solution'].any():
    valid = score_df[score_df['has_solution']].copy()
    best_per_instance = valid.groupby('instance')['cost'].min().rename('best_cost')
    valid = valid.merge(best_per_instance, on='instance', how='left')
    valid['gap_pct_to_best'] = 100.0 * (valid['cost'] - valid['best_cost']) / valid['best_cost']

    print('--- Objective comparison (lower is better) ---')
    display(valid.sort_values(['instance', 'cost']))

    summary_quality = valid.groupby('method').agg(
        solved_instances=('instance', 'nunique'),
        mean_cost=('cost', 'mean'),
        median_cost=('cost', 'median'),
        mean_gap_pct=('gap_pct_to_best', 'mean'),
        median_gap_pct=('gap_pct_to_best', 'median'),
    ).sort_values('mean_gap_pct')
    print('\n--- Quality summary by method ---')
    display(summary_quality)

    # Runtime comparison from exact benchmark records (if present)
    bench_file = Path('logs') / 'runtime_benchmarks.jsonl'
    if bench_file.exists():
        exact = pd.read_json(bench_file, lines=True)
        exact_ok = exact[(exact['returncode'] == 0) & (~exact['timed_out'])].copy()
        exact_ok = exact_ok[exact_ok['instance'].isin(COMPARE_INSTANCES) & exact_ok['method'].isin(COMPARE_METHODS)]
        if not exact_ok.empty:
            rt = exact_ok.groupby(['instance', 'method'])['wall_time_sec'].mean().reset_index()
            rt_summary = exact_ok.groupby('method')['wall_time_sec'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).sort_values('mean')
            print('\n--- Runtime summary by method (exact wall-clock) ---')
            display(rt_summary)

            merged = valid.merge(rt, on=['instance', 'method'], how='left')
            print('\n--- Combined quality + runtime view ---')
            display(merged.sort_values(['instance', 'gap_pct_to_best', 'wall_time_sec']))
        else:
            print('\nNo exact runtime records for selected methods/instances yet.')
    else:
        print('\nNo runtime benchmark file found yet (logs/runtime_benchmarks.jsonl).')

    # Plot: mean quality gap and mean runtime by method
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    qplot = summary_quality.reset_index()
    axes[0].bar(qplot['method'], qplot['mean_gap_pct'])
    axes[0].set_title('Mean quality gap to best (%)')
    axes[0].set_ylabel('Gap % (lower is better)')
    axes[0].tick_params(axis='x', rotation=30)

    if (Path('logs') / 'runtime_benchmarks.jsonl').exists():
        exact = pd.read_json(Path('logs') / 'runtime_benchmarks.jsonl', lines=True)
        exact_ok = exact[(exact['returncode'] == 0) & (~exact['timed_out'])].copy()
        exact_ok = exact_ok[exact_ok['instance'].isin(COMPARE_INSTANCES) & exact_ok['method'].isin(COMPARE_METHODS)]
        if not exact_ok.empty:
            rplot = exact_ok.groupby('method')['wall_time_sec'].mean().reset_index()
            axes[1].bar(rplot['method'], rplot['wall_time_sec'])
            axes[1].set_title('Mean runtime (s, exact)')
            axes[1].set_ylabel('Seconds (lower is better)')
            axes[1].tick_params(axis='x', rotation=30)
        else:
            axes[1].text(0.5, 0.5, 'No exact runtime records yet', ha='center', va='center')
            axes[1].set_axis_off()
    else:
        axes[1].text(0.5, 0.5, 'No exact runtime file yet', ha='center', va='center')
        axes[1].set_axis_off()

    plt.tight_layout()
else:
    print('No solution files found yet for selected methods/instances. Run methods first, then re-run this cell.')